# Social media growth pipeline — nonprofit growth analyst

**Data:** `social_media_posts` joined to `donations` on `referral_post_id` → `post_id`.

This notebook profiles what drives **referrals and revenue**, **when** to post, **boosted vs organic** performance, and patterns for **engagement** vs **fundraising** — then turns findings into strategy, a posting playbook, and anti-patterns.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_DIR = Path("lighthouse_csv_v7")
posts = pd.read_csv(DATA_DIR / "social_media_posts.csv", parse_dates=["created_at"])
donations = pd.read_csv(DATA_DIR / "donations.csv", parse_dates=["donation_date"])


def donation_value_php(row: pd.Series) -> float:
    if row["donation_type"] == "Monetary" and pd.notna(row["amount"]):
        return float(row["amount"])
    if row["donation_type"] == "InKind" and pd.notna(row["estimated_value"]):
        return float(row["estimated_value"])
    return 0.0


donations = donations.copy()
donations["value_php"] = donations.apply(donation_value_php, axis=1)
ref = donations[donations["referral_post_id"].notna()].copy()
ref["referral_post_id"] = ref["referral_post_id"].astype(int)
donations_by_post = ref.groupby("referral_post_id", as_index=False).agg(
    n_referral_gifts=("donation_id", "count"),
    referral_php=("value_php", "sum"),
)
donations_by_post = donations_by_post.rename(columns={"referral_post_id": "post_id"})

p = posts.merge(donations_by_post, on="post_id", how="left")
p["has_referral"] = p["n_referral_gifts"].fillna(0) > 0
p["referral_php"] = p["referral_php"].fillna(0)
p["n_referral_gifts"] = p["n_referral_gifts"].fillna(0)

p["cta_label"] = p["call_to_action_type"].fillna("(no CTA type)").replace("", "(no CTA type)")

print("Posts:", len(p), "| Referral-linked donation rows:", len(ref))
print("Posts with ≥1 attributed gift:", p["has_referral"].sum())
pd.DataFrame(
    {"value": [p["engagement_rate"].mean(), p["donation_referrals"].sum(), p["estimated_donation_value_php"].sum()]},
    index=["Mean engagement_rate (all posts)", "Sum donation_referrals (platform)", "Sum estimated_donation_value_php"],
)

## 1. Post characteristics that drive donations (attributed gifts)

We compare **share of referral PHP** and **mean post-level metrics** by dimension. For fundraising, prioritize categories with **high total `referral_php`** or **lift** vs volume share.

In [ ]:
def lift_table(df: pd.DataFrame, col: str, value_col: str = "referral_php"):
    tot = df[value_col].sum()
    g = df.groupby(col, dropna=False).agg(
        posts=("post_id", "count"),
        referral_php_sum=(value_col, "sum"),
        referral_gifts=("n_referral_gifts", "sum"),
        mean_engagement=("engagement_rate", "mean"),
        mean_platform_referrals=("donation_referrals", "mean"),
        mean_est_value=("estimated_donation_value_php", "mean"),
    )
    g["share_posts"] = g["posts"] / len(df)
    g["share_referral_php"] = g["referral_php_sum"] / tot if tot > 0 else 0
    g["lift_php"] = g["share_referral_php"] / g["share_posts"].replace(0, np.nan)
    return g.sort_values("referral_php_sum", ascending=False)


dims = [
    "platform",
    "post_type",
    "media_type",
    "sentiment_tone",
    "cta_label",
]
for col in dims:
    print("===", col, "===")
    display(lift_table(p, col).round(4))

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.ravel()
for ax, col in zip(axes, dims):
    t = lift_table(p, col)
    top = t.head(8)
    top["referral_php_sum"].plot(kind="barh", ax=ax, color="#2E86AB")
    ax.set_title(f"Referral PHP by {col} (top 8)")
plt.tight_layout()
plt.show()

## 2. Timing — best `day_of_week` and `post_hour`

We use **mean** post-level metrics (engagement, platform referrals, est. value) by slot. Interpret with caution: confounded by content mix.

In [ ]:
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
p["day_of_week"] = pd.Categorical(p["day_of_week"], categories=day_order, ordered=True)

by_day = (
    p.groupby("day_of_week", observed=False)
    .agg(
        n=("post_id", "count"),
        engagement=("engagement_rate", "mean"),
        don_referrals=("donation_referrals", "mean"),
        est_don=("estimated_donation_value_php", "mean"),
        referral_php=("referral_php", "sum"),
    )
    .reindex(day_order)
)
print("By day of week:")
display(by_day.round(4))

by_hour = (
    p.groupby("post_hour")
    .agg(
        n=("post_id", "count"),
        engagement=("engagement_rate", "mean"),
        don_referrals=("donation_referrals", "mean"),
        est_don=("estimated_donation_value_php", "mean"),
        referral_php=("referral_php", "sum"),
    )
    .sort_values("post_hour")
)
print("Hours with at least 15 posts (stable averages):")
display(by_hour[by_hour["n"] >= 15].sort_values("engagement", ascending=False).head(12).round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
by_day["engagement"].plot(kind="bar", ax=axes[0], color="#3A7D44")
axes[0].set_title("Mean engagement_rate by day")
axes[0].tick_params(axis="x", rotation=45)
by_hour[by_hour["n"] >= 10]["engagement"].plot(ax=axes[1], color="#A23B72", marker="o")
axes[1].set_title("Mean engagement_rate by hour (n≥10)")
axes[1].set_xlabel("Hour")
plt.tight_layout()
plt.show()

heat = p.pivot_table(
    values="engagement_rate",
    index="post_hour",
    columns="day_of_week",
    aggfunc="mean",
)
heat = heat.reindex(columns=day_order)
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(heat.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
ax.set_title("Engagement rate heatmap: hour × day (mean)")
plt.colorbar(im, ax=ax, label="engagement_rate")
plt.tight_layout()
plt.show()

## 3. Boosted vs organic (`is_boosted`)

Compare **reach**, **engagement**, **platform-reported referrals**, and **estimated donation value** — boosted posts often have different creative and budget.

In [ ]:
boost = p.groupby("is_boosted").agg(
    posts=("post_id", "count"),
    mean_reach=("reach", "mean"),
    mean_impressions=("impressions", "mean"),
    mean_engagement=("engagement_rate", "mean"),
    sum_platform_referrals=("donation_referrals", "sum"),
    mean_platform_referrals=("donation_referrals", "mean"),
    sum_est_don=("estimated_donation_value_php", "sum"),
    mean_est_don=("estimated_donation_value_php", "mean"),
    attributed_referral_php=("referral_php", "sum"),
    mean_budget=("boost_budget_php", "mean"),
)
display(boost.round(2))

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
boost["mean_engagement"].plot(kind="bar", ax=axes[0], color=["#888", "#F4A261"])
axes[0].set_title("Mean engagement_rate")
boost["mean_est_don"].plot(kind="bar", ax=axes[1], color=["#888", "#2E86AB"])
axes[1].set_title("Mean est. donation value (PHP)")
boost["attributed_referral_php"].plot(kind="bar", ax=axes[2], color=["#888", "#3A7D44"])
axes[2].set_title("Sum attributed referral PHP")
for ax in axes:
    ax.set_xticklabels(["Organic", "Boosted"], rotation=0)
plt.tight_layout()
plt.show()

## 4. Patterns that maximize engagement, referrals, and estimated value

We rank **top decile** of posts on each KPI and show **over-represented** `post_type` / `platform` vs baseline.

In [ ]:
def top_decile_profile(df: pd.DataFrame, kpi: str, label: str):
    q = df[kpi].quantile(0.9)
    top = df[df[kpi] >= q]
    base_pt = df["post_type"].value_counts(normalize=True)
    top_pt = top["post_type"].value_counts(normalize=True)
    lift_pt = (top_pt / base_pt.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).dropna()
    lift_pt = lift_pt.sort_values(ascending=False).head(8)
    base_pl = df["platform"].value_counts(normalize=True)
    top_pl = top["platform"].value_counts(normalize=True)
    lift_pl = (top_pl / base_pl.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).dropna()
    lift_pl = lift_pl.sort_values(ascending=False).head(8)
    print(f"--- Top decile on {label} (threshold {kpi} >= {q:.4f}) — n={len(top)} ---")
    print("Post type lift:")
    display(lift_pt.round(2).to_frame("lift_vs_share"))
    print("Platform lift:")
    display(lift_pl.round(2).to_frame("lift_vs_share"))


top_decile_profile(p, "engagement_rate", "engagement_rate")
top_decile_profile(p, "donation_referrals", "donation_referrals")
top_decile_profile(p, "estimated_donation_value_php", "estimated_donation_value_php")

corr = p[
    [
        "engagement_rate",
        "donation_referrals",
        "estimated_donation_value_php",
        "referral_php",
        "reach",
        "num_hashtags",
        "caption_length",
    ]
].corr(numeric_only=True)
print("Correlation matrix (KPIs vs scale):")
display(corr.round(3))

---

## Growth strategy (nonprofit + growth)

- **Double down on formats that win both *share of attributed PHP* and *lift* in section 1** — especially `post_type` / `platform` combos where you punch above your post-count weight.
- **Separate goals:** high **engagement_rate** does not always equal **donations**; allocate a **fundraising** track (clear CTA, urgency, impact story) vs a **community** track (education, trust-building) and measure each separately.
- **Timing:** use section 2 slots to **schedule** tests; run A/Bs holding `post_type` constant so day/hour effects are real.
- **Boosts:** treat paid as **amplification of proven organic winners** — boost posts that already show referral or strong est. value, not cold experiments (unless testing audiences).
- **Attribution hygiene:** keep `donation_referrals` / CRM linkages consistent so you can learn which posts actually fund programs, not only platform estimates.

---

## Optimal posting playbook (90-day)

| Week | Focus | Action |
|------|--------|--------|
| 1–2 | **Baseline** | Export top 20% posts by `referral_php` + `engagement_rate`; tag creative pattern (CTA, story, media). |
| 3–4 | **Timing** | Post 50% of fundraising content in **best hours** from your heatmap; keep content identical for comparability. |
| 5–6 | **Format** | Run 2–3 **ImpactStory** / **FundraisingAppeal** variants per platform; standardize CTA (`DonateNow` vs `LearnMore`). |
| 7–8 | **Boost** | Put **10–20% of boost budget** behind top organic referral posts; compare CPA vs cold boost. |
| 9–12 | **Scale** | Weekly cadence: 2× story/education, 1× direct ask, 1× thank-you/progress; monthly review lift tables. |

---

## What **NOT** to do (anti-patterns)

- **Do not** optimize only **engagement** — vanity metrics can crowd out **donation** and **referral** KPIs.
- **Do not** boost **every** post — dilutes learning and burns budget; boost **after** organic proof.
- **Do not** ignore **CTA** and **sentiment** — missing or vague CTAs and mismatched tone reduce conversion on asks.
- **Do not** post **high-frequency** low-value updates without a **content calendar** tied to campaigns — creates noise and hides attribution.
- **Do not** compare platforms **without normalizing** for audience size and organic reach — use **reach-weighted** or **per-follower** metrics when possible.
- **Do not** treat **estimated_donation_value_php** as cash — reconcile with **CRM `referral_php`** to avoid overconfidence.

---

**Growth-hacker mindset:** ship small experiments, read **lift** not just averages, and **instrument** every ask with UTM + `referral_post_id` so next quarter’s playbook is smarter than this one.